# Student Project and Skill Tracking System (SPSTS)
## Exploratory Data Analysis & Statistical Visualizations
---
- **Student Name:** Gokul . S
- **Register Number:** 732925ITR032
- **Class:** 25IT3A
- **Department:** Information Technology
- **Institution:** Velalar College of Engineering and Technology, Erode - 12
- **Course:** 25ITC31 - Foundations of Data Science


### 1. Import Libraries and Connect to Database
*(Covers Unit 2: Python Foundations & File Handling, Unit 3: Data Wrangling)*

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
conn = sqlite3.connect("../database/student_tracking.db")

# Read tables into Pandas DataFrames
df_students = pd.read_sql_query("SELECT * FROM students", conn)
df_skills = pd.read_sql_query("SELECT * FROM skills", conn)
df_student_skills = pd.read_sql_query("SELECT * FROM student_skills", conn)
df_projects = pd.read_sql_query("SELECT * FROM projects", conn)
df_milestones = pd.read_sql_query("SELECT * FROM milestones", conn)
df_evaluations = pd.read_sql_query("SELECT * FROM evaluations", conn)
conn.close()

print(f"Loaded {len(df_students)} students, {len(df_projects)} projects, {len(df_evaluations)} evaluations.")

### 2. Data Cleaning & Transformation (Unit 4)
- Handling missing dates, converting string formats to datetime objects.
- Feature engineering: Calculating milestone turnaround days and project completion rates.

In [2]:
# Handling missing submission dates and converting to datetime
df_milestones["submission_date_clean"] = df_milestones["submission_date"].fillna("Pending / Not Submitted")
df_milestones["due_date_dt"] = pd.to_datetime(df_milestones["due_date"])
df_milestones["submission_date_dt"] = pd.to_datetime(df_milestones["submission_date"], errors="coerce")

# Turnaround time in days
df_milestones["turnaround_days"] = (df_milestones["submission_date_dt"] - df_milestones["due_date_dt"]).dt.days

# Merge student details with skills
df_skills_merged = df_student_skills.merge(df_students, on="register_no").merge(df_skills, on="skill_id")
df_skills_merged.head()

### 3. Descriptive Statistics & Summary (Unit 1 & Unit 5)
- Measures of Central Tendency (Mean, Median) and Variability (Standard Deviation, IQR).

In [3]:
eval_stats = df_evaluations["total_marks"].describe()
print("Evaluation Rubric Total Score Statistics (Max 100):")
print(eval_stats)

### 4. Visualizations & Patterns (Unit 5)
#### Chart 1: Project Completion Rates (Donut Chart)

In [4]:
plt.figure(figsize=(7, 7))
status_counts = df_projects["completion_status"].value_counts()
colors = ["#2ecc71", "#3498db", "#e74c3c", "#f1c40f"]
plt.pie(status_counts, labels=status_counts.index, autopct="%1.1f%%", startangle=140, colors=colors[:len(status_counts)], wedgeprops=dict(width=0.4, edgecolor="white"))
plt.title("Student Project Completion Status", fontsize=13, weight="bold")
plt.show()

#### Chart 2: Popular Technical Skills & Choices

In [5]:
plt.figure(figsize=(10, 5))
top_skills = df_skills_merged["skill_name"].value_counts().head(8)
sns.barplot(x=top_skills.values, y=top_skills.index, hue=top_skills.index, palette="viridis", legend=False)
plt.title("Most Popular Technical Skills among Students", fontsize=13, weight="bold")
plt.xlabel("Count of Students", weight="bold")
plt.ylabel("Skill Name", weight="bold")
plt.show()

#### Chart 3: Evaluation Component Mark Distributions (Box Plots)

In [6]:
rubric_cols = ["literature_survey_marks", "system_design_marks", "implementation_marks", "testing_marks", "presentation_marks"]
plt.figure(figsize=(11, 5))
sns.boxplot(data=df_evaluations[rubric_cols], palette="Set2")
plt.xticks(ticks=range(5), labels=["Survey (20)", "Design (20)", "Impl (30)", "Testing (20)", "Viva (10)"], weight="bold")
plt.title("Assessment Component Rubric Score Ranges", fontsize=13, weight="bold")
plt.ylabel("Marks Awarded", weight="bold")
plt.show()